In [0]:
%run  "/Workspace/Users/sanskruti.r.sampate@v4c.ai/Vstone_flight/notebooks/00_configs" 

In [0]:
dbutils.widgets.dropdown(
    "silver_table",
    "flight_silver",
    [
        "flight_silver",
        "flight_json_silver",
        "flight_xml_silver",
        "flight_csv_incremental_silver"
    ]
)

silver_table = dbutils.widgets.get("silver_table")
silver_table_full_name = f"{catalog}.{silver_schema}.{silver_table}"

read silver table


In [0]:
silver_df = spark.read.table(silver_table_full_name)

print("Rows :", silver_df.count())
print("Columns :", len(silver_df.columns))

display(silver_df.limit(5))

In [0]:
#unit test : table exists

assert spark.catalog.tableExists(silver_table_full_name)
print(" TEST PASSED : Silver table exists")

In [0]:
#unit test : row records

assert silver_df.count() > 0
print(" TEST PASSED : Table contains records")

In [0]:
#unit test : required ccolumns exists

required_columns = [
    "flightdate",
    "airline",
    "origin",
    "dest",
    "distance",
    "processed_timestamp",
    "pipeline_layer"
]

missing = []

for c in required_columns:
    if c not in silver_df.columns:
        missing.append(c)

assert len(missing) == 0

print(" TEST PASSED")
print("Missing Columns :", missing)

In [0]:
#unit test : column names standardized

bad_columns = [
    c for c in silver_df.columns
    if " " in c or any(x.isupper() for x in c)
]

assert len(bad_columns) == 0

print(" TEST PASSED")

In [0]:
#unit test : duplicate records removed

business_keys = [
    "flightdate",
    "airline",
    "origin",
    "dest",
    "crsdeptime"
]

duplicates = (
    silver_df
    .groupBy(business_keys)
    .count()
    .filter("count > 1")
    .count()
)

assert duplicates == 0

print("TEST PASSED : No duplicate records")

In [0]:
#unit test : critical columns have no nulls

from pyspark.sql import functions as F

display(

silver_df.select(

F.count(F.when(F.col("flightdate").isNull(),1)).alias("flightdate"),

F.count(F.when(F.col("airline").isNull(),1)).alias("airline"),

F.count(F.when(F.col("origin").isNull(),1)).alias("origin"),

F.count(F.when(F.col("dest").isNull(),1)).alias("dest")

)

)

In [0]:
assert silver_df.filter(

F.col("flightdate").isNull() |
F.col("airline").isNull() |
F.col("origin").isNull() |
F.col("dest").isNull()

).count()==0

print(" TEST PASSED")

In [0]:
#unit test : date standardization

assert dict(silver_df.dtypes)["flightdate"] == "date"

print(" TEST PASSED : flightdate is DateType")

In [0]:
#unit test : audit columns

assert "processed_timestamp" in silver_df.columns
assert "pipeline_layer" in silver_df.columns

print(" TEST PASSED")

In [0]:
#unit test : negative values

assert silver_df.filter(F.col("distance")<0).count()==0

assert silver_df.filter(F.col("airtime")<0).count()==0

print(" TEST PASSED")

#delta lake time travel demonstration:

# Delta Lake Time Travel

Delta Lake maintains version history for every transaction.

Benefits:

- Recover accidentally deleted data
- Audit historical changes
- Compare different versions
- Support reproducibility

Evidence:

- DESCRIBE HISTORY output
- Version 0 data
- Current version data

In [0]:
#time travel:

spark.sql(f"DESCRIBE HISTORY {silver_table_full_name}").display()

In [0]:
spark.sql(f"describe history {silver_table_full_name}").display

In [0]:
spark.sql(f"""
          select * from {silver_table_full_name}
          version as of 0
          """).display()